# OS Tutor: QLoRA fine-tuning of Llama 3.1 8B

Fine-tunes Llama 3.1 8B Instruct on 1216 tutoring examples (explanations, engine-verified worked solutions,
answer checking, quizzes, teach-back feedback, class-notes Q&A), merges the LoRA adapter into the model and exports
one GGUF file (~4.9 GB) for Ollama. (Recent Ollama versions no longer load LoRA adapters, hence the merged file.)

**Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Allow Google Drive access when
asked at the end: the model file is copied to *My Drive* as `os-tutor.gguf` (needs ~5 GB free in Drive).
**Kaggle:** Settings → Accelerator **GPU T4** and **Internet on**, then **Run all**. Download `os-tutor.gguf`
from the Output panel.

Takes about 60–80 minutes on a T4 (install ~3 min, model download ~3 min, training ~35–45 min, export ~20 min).
Then on your laptop: save the file as `qlora\os-tutor.gguf` and run `qlora\setup_ollama_models.bat`.
If the session ends before the export, don't retrain: use `export_gguf_colab.ipynb` with `os-tutor-lora.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth

## 1. Training data (built in)

In [ ]:
import base64, gzip, json, os, pathlib
DATA = "..."
raw = gzip.decompress(base64.b64decode(DATA)).decode("utf-8")
train_txt, val_txt = raw.split("\n=====SPLIT=====\n")
pathlib.Path("data").mkdir(exist_ok=True)
open("data/train.jsonl", "w", encoding="utf-8").write(train_txt)
open("data/val.jsonl", "w", encoding="utf-8").write(val_txt)
print(len(train_txt.splitlines()), "training examples,", len(val_txt.splitlines()), "validation examples")
print(json.loads(train_txt.splitlines()[0])["messages"][1]["content"][:300])

## 2. Settings

In [ ]:
import torch
EPOCHS = 1            # 1 is enough to change the style; 2 fits the data more closely (twice the time)
MAX_SEQ = 2048
LORA_R = 16
LR = 2e-4
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH, ACCUM = (2, 4) if gpu_gb > 12 else (1, 8)      # effective batch 8 either way
print(f"GPU memory {gpu_gb:.0f} GB -> batch {BATCH} x accumulation {ACCUM}")

## 3. Load Llama 3.1 8B in 4-bit and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
model.print_trainable_parameters()

## 4. Train (loss only on the tutor's replies)

In [ ]:
import inspect
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

ds = load_dataset("json", data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"})
def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}
ds = ds.map(to_text, batched=True, remove_columns=ds["train"].column_names)

cfg = dict(dataset_text_field="text", per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
           warmup_steps=10, num_train_epochs=EPOCHS, learning_rate=LR, logging_steps=10, optim="adamw_8bit",
           weight_decay=0.01, lr_scheduler_type="linear", seed=3407, bf16=is_bfloat16_supported(),
           fp16=not is_bfloat16_supported(), output_dir="checkpoints", report_to="none", save_strategy="no")
names = inspect.signature(SFTConfig.__init__).parameters
cfg["max_seq_length" if "max_seq_length" in names else "max_length"] = MAX_SEQ
common = dict(model=model, train_dataset=ds["train"], eval_dataset=ds["validation"], args=SFTConfig(**cfg))
try:
    trainer = SFTTrainer(tokenizer=tokenizer, **common)
except TypeError:
    trainer = SFTTrainer(processing_class=tokenizer, **common)
trainer = train_on_responses_only(trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n")
stats = trainer.train()
print(stats)
print(trainer.evaluate())

## 5. Try it

In [ ]:
FastLanguageModel.for_inference(model)
SYSTEM = "You are OS Tutor, a friendly tutor for the VIT BCSE303 Operating Systems course. You teach CPU scheduling and process synchronization (critical sections, semaphores, classic problems, deadlocks, the Banker's algorithm). Explain from scratch: an everyday example first, then the exact rule. Follow the class-notes conventions: ties go to the earlier arrival; a lower priority number means higher priority unless the question says otherwise; in Round Robin a process arriving when a slice ends joins the queue before the preempted one; TAT = CT \u2212 AT, WT = TAT \u2212 BT, RT = first start \u2212 AT. When a student shows you their answer, find the first mistake and give a hint before giving the answer away. Keep replies focused, and end with one short question or next step unless the student only wanted a fact."
for q in ["Explain SRTF to me like I'm new to it",
          "In round robin, if a new process arrives exactly when the quantum ends, who goes first?",
          "Quiz me on deadlocks"]:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=350, temperature=0.3, do_sample=True)
    print("\n>>>", q, "\n" + tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 6. Save the adapter (small backup, ~160 MB)

In [ ]:
import shutil
model.save_pretrained("os-tutor-lora")                  # adapter only
tokenizer.save_pretrained("os-tutor-lora")              # so the zip is self-contained
shutil.make_archive("os-tutor-lora", "zip", "os-tutor-lora")
print(sorted(os.listdir("os-tutor-lora")))
if os.path.exists("/kaggle/working"):
    shutil.copy("os-tutor-lora.zip", "/kaggle/working/os-tutor-lora.zip")
else:
    from google.colab import files
    files.download("os-tutor-lora.zip")

## 7. Merge and export the model for Ollama (GGUF, ~15–25 minutes)
Merges the adapter into Llama 3.1 8B and quantizes to Q4_K_M (~4.9 GB), the same size class as `llama3.1:8b`.

In [ ]:
DO_GGUF = True
if DO_GGUF:
    model.save_pretrained_gguf("os-tutor-gguf", tokenizer, quantization_method="q4_k_m")
    print("export finished")

## 8. Copy the model file to Google Drive

In [ ]:
import os, shutil
def find_q4():
    hits = []
    for root, dirs, files in os.walk("."):
        dirs[:] = [d for d in dirs if d not in ("drive", "sample_data") and not d.startswith(".")]
        hits += [os.path.join(root, f) for f in files if f.lower().endswith(".gguf") and "q4_k_m" in f.lower()]
    return hits
hits = find_q4()
assert hits, "No Q4_K_M .gguf file found. Did the export cell above finish without an error?"
src = max(hits, key=os.path.getsize)
print("model file:", src, f"({os.path.getsize(src) / 1e9:.1f} GB)")
if os.path.exists("/kaggle/working"):
    shutil.copy(src, "/kaggle/working/os-tutor.gguf")
    print("Kaggle: download os-tutor.gguf from the Output panel on the right.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(src, "/content/drive/MyDrive/os-tutor.gguf")
    print("Copied to Google Drive -> My Drive -> os-tutor.gguf")
    print("Download it, save it on your laptop as qlora\\os-tutor.gguf and run qlora\\setup_ollama_models.bat")